# Notebook 09 Part 1A: Reading and Writing CSV & JSON in PySpark


This notebook focuses on the two most common source formats:

```text
CSV  → flat tabular files, Excel exports, reports
JSON → APIs, logs, events, nested/semi-structured data
```

## 1. Windows Local Setup

We set HADOOP_HOME in local PySpark on Windows because Spark uses Hadoop libraries internally for file-system operations. Even when we are not running a Hadoop cluster, Spark still depends on Hadoop’s file-system layer to read and write files such as CSV, JSON, and Parquet.

In [1]:
from pathlib import Path


In [2]:
print(Path("C:/hadoop/bin/winutils.exe").exists())
print(Path("C:/hadoop/bin/hadoop.dll").exists())

True
True


* create ``` C:/hadoop/bin/ ```
* download ```winutils.exe``` and ```hadoop.dll``` file from ```https://github.com/cdarlint/winutils/tree/master```
* check version of spark from below command 

In [3]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


**Set up Hadoop path**

In [4]:
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]

## 2. Create SparkSession

In [5]:

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("CSV_JSON_File_Handling") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

'3.4.2'

In [6]:
print(Path("C:/hadoop/bin/winutils.exe").exists())
print(Path("C:/hadoop/bin/hadoop.dll").exists())

True
True


## 3. Create Demo Folder

In [7]:


base_path = Path("pyspark_csv_json_demo")
base_path.mkdir(exist_ok=True)

print("Demo folder:", base_path.resolve())

Demo folder: E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_csv_json_demo


# CSV SECTION

## 4. Create a Sample CSV File

In [7]:
sales_csv = base_path / "sales.csv"

sales_csv.write_text(
'''order_id,customer_id,customer_name,city,product,quantity,amount
101,1,Amit,Delhi,Laptop,1,55000
102,2,Priya,Mumbai,Mobile,2,40000
103,3,Rahul,Delhi,Tablet,1,25000
104,4,Sneha,Pune,Laptop,1,60000
105,5,Karan,Mumbai,Monitor,2,30000
106,6,Meera,Delhi,Mobile,,22000
''',
encoding="utf-8"
)

print(sales_csv.resolve())

E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_csv_json_demo\sales.csv


## 5. Read CSV Without Options

In [8]:
df_basic = spark.read.csv("E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/sales.csv")
df_basic.show()

+--------+-----------+-------------+------+-------+--------+------+
|     _c0|        _c1|          _c2|   _c3|    _c4|     _c5|   _c6|
+--------+-----------+-------------+------+-------+--------+------+
|order_id|customer_id|customer_name|  city|product|quantity|amount|
|     101|          1|         Amit| Delhi| Laptop|       1| 55000|
|     102|          2|        Priya|Mumbai| Mobile|       2| 40000|
|     103|          3|        Rahul| Delhi| Tablet|       1| 25000|
|     104|          4|        Sneha|  Pune| Laptop|       1| 60000|
|     105|          5|        Karan|Mumbai|Monitor|       2| 30000|
|     106|          6|        Meera| Delhi| Mobile|    NULL| 22000|
+--------+-----------+-------------+------+-------+--------+------+



Spark gives default column names like `_c0`, `_c1`, `_c2` because we did not tell Spark that the first row contains headers.

## 6. Read CSV with `header=True`

In [9]:
df_header = spark.read.csv(
    "E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/sales.csv",
    header=True
)

df_header.show()
df_header.printSchema()

+--------+-----------+-------------+------+-------+--------+------+
|order_id|customer_id|customer_name|  city|product|quantity|amount|
+--------+-----------+-------------+------+-------+--------+------+
|     101|          1|         Amit| Delhi| Laptop|       1| 55000|
|     102|          2|        Priya|Mumbai| Mobile|       2| 40000|
|     103|          3|        Rahul| Delhi| Tablet|       1| 25000|
|     104|          4|        Sneha|  Pune| Laptop|       1| 60000|
|     105|          5|        Karan|Mumbai|Monitor|       2| 30000|
|     106|          6|        Meera| Delhi| Mobile|    NULL| 22000|
+--------+-----------+-------------+------+-------+--------+------+

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: string (nullable = true)



Now column names are correct, but columns may still be read as strings because CSV is plain text.

## 7. Read CSV with `inferSchema=True`

In [10]:
df_csv = spark.read.csv(
    "E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/sales.csv",
    header=True,
    inferSchema=True
)

df_csv.show()
df_csv.printSchema()

+--------+-----------+-------------+------+-------+--------+------+
|order_id|customer_id|customer_name|  city|product|quantity|amount|
+--------+-----------+-------------+------+-------+--------+------+
|     101|          1|         Amit| Delhi| Laptop|       1| 55000|
|     102|          2|        Priya|Mumbai| Mobile|       2| 40000|
|     103|          3|        Rahul| Delhi| Tablet|       1| 25000|
|     104|          4|        Sneha|  Pune| Laptop|       1| 60000|
|     105|          5|        Karan|Mumbai|Monitor|       2| 30000|
|     106|          6|        Meera| Delhi| Mobile|    NULL| 22000|
+--------+-----------+-------------+------+-------+--------+------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: integer (nullable = true)



`inferSchema=True` asks Spark to scan data and guess data types.

It is easy for learning. For production pipelines, a custom schema is usually safer and more controlled.

## 8. Recommended Option Style

In [11]:
df_csv_option_style = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_csv_json_demo\sales.csv")

df_csv_option_style.show()

+--------+-----------+-------------+------+-------+--------+------+
|order_id|customer_id|customer_name|  city|product|quantity|amount|
+--------+-----------+-------------+------+-------+--------+------+
|     101|          1|         Amit| Delhi| Laptop|       1| 55000|
|     102|          2|        Priya|Mumbai| Mobile|       2| 40000|
|     103|          3|        Rahul| Delhi| Tablet|       1| 25000|
|     104|          4|        Sneha|  Pune| Laptop|       1| 60000|
|     105|          5|        Karan|Mumbai|Monitor|       2| 30000|
|     106|          6|        Meera| Delhi| Mobile|    NULL| 22000|
+--------+-----------+-------------+------+-------+--------+------+



## 9. Define Custom Schema

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

sales_schema = StructType([
    StructField("order_id", IntegerType(), True), # Nullable = True, null values allowed
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True)
])

## 10. Read CSV with Custom Schema

In [13]:
df_custom = spark.read \
    .option("header", True) \
    .schema(sales_schema) \
    .csv("E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/sales.csv")

df_custom.show()
df_custom.printSchema()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     106|          6|        Meera| Delhi| Mobile|    NULL|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)



## 11. `inferSchema` vs Custom Schema

| Method | Advantage | Limitation |
|---|---|---|
| No schema | Very simple | Most columns become strings |
| `inferSchema=True` | Easy for learning | Spark must inspect data |
| Custom schema | Controlled, predictable, production-friendly | Schema must be written manually |

## 12. Read CSV with Different Delimiter

In [15]:
base_path = Path("E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo")
pipe_file = base_path   / "employees_pipe.csv"

pipe_file.write_text(
'''employee_id|employee_name|department|salary
1|Amit|Sales|45000
2|Priya|IT|65000
3|Rahul|HR|50000
''',
encoding="utf-8"
)

employees_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("delimiter", "|") \
    .csv(str(pipe_file)) # passing path

employees_df.show()

+-----------+-------------+----------+------+
|employee_id|employee_name|department|salary|
+-----------+-------------+----------+------+
|          1|         Amit|     Sales| 45000|
|          2|        Priya|        IT| 65000|
|          3|        Rahul|        HR| 50000|
+-----------+-------------+----------+------+



Common delimiters:

| Delimiter | Option value |
|---|---|
| Comma | `,` |
| Pipe | `|` |
| Semicolon | `;` |
| Tab | `\t` |

## 13. Handle Custom Null Values

In [50]:
null_file = base_path / "sales_with_nulls.csv"

null_file.write_text(
'''order_id,customer,amount
1,Amit,5000
2,Priya,NA
3,Rahul,NULL
4,Sneha,
''',
encoding="utf-8"
)
  # it will treat "NA" as Null
df_nulls = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("nullValue", "NA") \
    .csv(str(null_file))

df_nulls.show()
df_nulls.printSchema()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|       1|    Amit|  5000|
|       2|   Priya|  NULL|
|       3|   Rahul|  NULL|
|       4|   Sneha|  NULL|
+--------+--------+------+

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: string (nullable = true)



Here only `NA` is treated as null because we passed:

```python
.option("nullValue", "NA")
```

For multiple null-like values, we should clean manually.

## 14. Handle Multiple Null-Like Values

In [17]:
from pyspark.sql.functions import col, when, trim

df_null_cleaned = spark.read \
    .option("header", True) \
    .csv(str(null_file))

df_null_cleaned = df_null_cleaned.withColumn(
    "amount",
    when(
        (col("amount").isNull()) |
        (trim(col("amount")) == "") |
        (col("amount").isin("NA", "NULL", "N/A")),
        None
    ).otherwise(col("amount").cast("double"))
)

df_null_cleaned.show()
df_null_cleaned.printSchema()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|       1|    Amit|5000.0|
|       2|   Priya|  NULL|
|       3|   Rahul|  NULL|
|       4|   Sneha|  NULL|
+--------+--------+------+

root
 |-- order_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- amount: double (nullable = true)



## 16. Quoted Values Containing Commas

In [18]:
quoted_file = base_path / "quoted_products.csv"

quoted_file.write_text(
'''product_id,product_name,description
1,Laptop,"High performance, business laptop"
2,Mobile,"Android phone, 5G support"
3,Monitor,"24 inch, Full HD"
''',
encoding="utf-8"
)

# quote " will allow comma between quotes
df_quoted = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("quote", '"') \
    .csv(str(quoted_file))

df_quoted.show(truncate=False)

+----------+------------+---------------------------------+
|product_id|product_name|description                      |
+----------+------------+---------------------------------+
|1         |Laptop      |High performance, business laptop|
|2         |Mobile      |Android phone, 5G support        |
|3         |Monitor     |24 inch, Full HD                 |
+----------+------------+---------------------------------+



The comma inside quotes is treated as part of the value, not as a delimiter.

## 17. Multiline CSV Records

In [19]:
multiline_csv = base_path / "multiline_notes.csv"

multiline_csv.write_text(
'''id,name,notes
1,Amit,"Good student
attends regularly"
2,Priya,"Excellent
in practical work"
''',
encoding="utf-8"
)

df_multiline_csv = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .csv(str(multiline_csv))

df_multiline_csv.show(truncate=False)

+---+-----+-------------------------------+
|id |name |notes                          |
+---+-----+-------------------------------+
|1  |Amit |Good student\nattends regularly|
|2  |Priya|Excellent\nin practical work   |
+---+-----+-------------------------------+



Use `multiLine=True` when one record can span multiple lines.

## 18. Malformed CSV Rows and Read Modes

In [51]:
bad_csv = base_path / "bad_sales.csv"

bad_csv.write_text(
'''order_id,customer,amount
1,Amit,5000
2,Priya,7000
bad_row_without_correct_columns
3,Rahul,9000
''',
encoding="utf-8"
)

bad_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("amount", DoubleType(), True)
])

df_bad_permissive = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .schema(bad_schema) \
    .csv(str(bad_csv))

df_bad_permissive.show()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|       1|    Amit|5000.0|
|       2|   Priya|7000.0|
|    NULL|    NULL|  NULL|
|       3|   Rahul|9000.0|
+--------+--------+------+



Common read modes:

| Mode | Meaning |
|---|---|
| `PERMISSIVE` | Keep rows and set problematic fields as null |
| `DROPMALFORMED` | Drop malformed rows |
| `FAILFAST` | Stop immediately when bad data is found |

## 19. Drop Malformed CSV Rows

In [21]:
df_bad_drop = spark.read \
    .option("header", True) \
    .option("mode", "DROPMALFORMED") \
    .schema(bad_schema) \
    .csv(str(bad_csv))

df_bad_drop.show()

+--------+--------+------+
|order_id|customer|amount|
+--------+--------+------+
|       1|    Amit|5000.0|
|       2|   Priya|7000.0|
|       3|   Rahul|9000.0|
+--------+--------+------+



In [27]:
daily_sales_path = base_path / "daily_sales"
daily_sales_path.mkdir(exist_ok=True)

(daily_sales_path / "sales_day1.csv").write_text(
'''order_id,city,amount
1,Delhi,1000
2,Mumbai,2000
''',
encoding="utf-8"
)

(daily_sales_path / "sales_day2.csv").write_text(
'''order_id,city,amount
3,Delhi,1500
4,Pune,3000
''',
encoding="utf-8"
)

daily_pattern_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(daily_sales_path / "*.csv"))

daily_pattern_df.show()

+--------+------+------+
|order_id|  city|amount|
+--------+------+------+
|       1| Delhi|  1000|
|       2|Mumbai|  2000|
|       3| Delhi|  1500|
|       4|  Pune|  3000|
+--------+------+------+



In [29]:
from pyspark.sql.functions import input_file_name

daily_with_file_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(str(daily_sales_path)) \
    .withColumn("source_file", input_file_name())

daily_with_file_df.show(truncate=False)

+--------+------+------+-----------------------------------------------------------------------------------------------------------+
|order_id|city  |amount|source_file                                                                                                |
+--------+------+------+-----------------------------------------------------------------------------------------------------------+
|1       |Delhi |1000  |file:///E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/daily_sales/sales_day1.csv|
|2       |Mumbai|2000  |file:///E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/daily_sales/sales_day1.csv|
|3       |Delhi |1500  |file:///E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/daily_sales/sales_day2.csv|
|4       |Pune  |3000  |file:///E:/data/Documents/Data_Engineering/Module4-Pyspark/pyspark_csv_json_demo/daily_sales/sales_day2.csv|
+--------+------+------+---------------------------------------------

`input_file_name()` helps in audit, debugging and file-level lineage.

## 23. Write DataFrame as CSV

In [52]:
csv_output_path = base_path / "output_sales_csv/"

df_custom.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(csv_output_path))

print(csv_output_path.resolve())

E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_csv_json_demo\output_sales_csv


## 24. Spark Output Folder and Part Files

In [33]:
for file in csv_output_path.iterdir():
    print(file.name)

.part-00000-3cb738c6-2dac-48ea-9b5e-9c323f4a2d81-c000.csv.crc
._SUCCESS.crc
part-00000-3cb738c6-2dac-48ea-9b5e-9c323f4a2d81-c000.csv
_SUCCESS


Spark writes a folder because multiple partitions can be written in parallel.

Actual data files usually start with:

**.crc** is checksum/helper files.

```text
part-
```

`_SUCCESS` only confirms successful write.

## 25. Write One CSV Part File for Small Demo

### To decrease number of partition

In [36]:
single_csv_output = base_path / "single_output_csv"

df_custom.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(single_csv_output))

for file in single_csv_output.iterdir():
    print(file.name)

.part-00000-c2bb2f8c-3749-43e1-ba7c-d926d5eb50e1-c000.csv.crc
._SUCCESS.crc
part-00000-c2bb2f8c-3749-43e1-ba7c-d926d5eb50e1-c000.csv
_SUCCESS


Use `coalesce(1)` only for small demo exports. Do not use it blindly on large data.
<br> **coalesce() is used to reduce the number of partitions in a PySpark DataFrame.**

### To increase number of partition

In [53]:
single_csv_output = base_path / "single_output_csv"

df_custom.repartition(2) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(single_csv_output))

for file in single_csv_output.iterdir():
    print(file.name)

.part-00000-30d05cd3-6c31-441a-ad90-5b229998e2c6-c000.csv.crc
.part-00001-30d05cd3-6c31-441a-ad90-5b229998e2c6-c000.csv.crc
._SUCCESS.crc
part-00000-30d05cd3-6c31-441a-ad90-5b229998e2c6-c000.csv
part-00001-30d05cd3-6c31-441a-ad90-5b229998e2c6-c000.csv
_SUCCESS


## 26. CSV Write Modes

| Mode | Meaning |
|---|---|
| `overwrite` | Replace existing output |
| `append` | Add new files/data |
| `ignore` | Do nothing if path exists |
| `error` | Raise error if path exists |

In [38]:
write_mode_csv_path = base_path / "write_mode_csv"

# limit is used to limit number of rows
df_custom.limit(2).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(write_mode_csv_path))

df_custom.limit(1).write \
    .mode("append") \
    .option("header", True) \
    .csv(str(write_mode_csv_path))

spark.read.option("header", True).csv(str(write_mode_csv_path)).show()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
+--------+-----------+-------------+------+-------+--------+-------+



## 28. Write CSV with Compression

In [54]:
compressed_csv_path = base_path / "compressed_csv_gzip"

df_custom.write \
    .mode("overwrite") \
    .option("header", True) \
    .option("compression", "gzip") \
    .csv(str(compressed_csv_path))

for file in compressed_csv_path.iterdir():
    print(file.name)

.part-00000-217941f1-c48c-4972-b253-06d2fde68d17-c000.csv.gz.crc
._SUCCESS.crc
part-00000-217941f1-c48c-4972-b253-06d2fde68d17-c000.csv.gz
_SUCCESS


## 29. Write CSV Partitioned by Column

In [40]:
partitioned_csv_path = base_path / "sales_csv_by_city"

df_custom.write \
    .mode("overwrite") \
    .option("header", True) \
    .partitionBy("city") \
    .csv(str(partitioned_csv_path))

for item in sorted(partitioned_csv_path.iterdir()):
    print(item.name)

._SUCCESS.crc
_SUCCESS
city=Delhi
city=Mumbai
city=Pune


# JSON SECTION

## 30. Create Line-Delimited JSON

In [9]:
json_file = base_path / "customers.json"

json_file.write_text(
'''{"customer_id": 1, "name": "Amit", "city": "Delhi", "age": 25}
{"customer_id": 2, "name": "Priya", "city": "Mumbai", "age": 28}
{"customer_id": 3, "name": "Rahul", "city": "Pune", "age": 30}
''',
encoding="utf-8"
)

191

## 31. Read Line-Delimited JSON

In [10]:
customers_json_df = spark.read.json(str(json_file))

customers_json_df.show()
customers_json_df.printSchema()

+---+------+-----------+-----+
|age|  city|customer_id| name|
+---+------+-----------+-----+
| 25| Delhi|          1| Amit|
| 28|Mumbai|          2|Priya|
| 30|  Pune|          3|Rahul|
+---+------+-----------+-----+

root
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)



JSON stores field names, so Spark can usually infer structure more naturally than CSV.

## 32. Read JSON with Custom Schema

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

In [9]:
customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("age", IntegerType(), True)
])

customers_json_schema_df = spark.read \
    .schema(customer_schema) \
    .json(str(json_file))

customers_json_schema_df.show()
customers_json_schema_df.printSchema()

+-----------+-----+------+---+
|customer_id| name|  city|age|
+-----------+-----+------+---+
|          1| Amit| Delhi| 25|
|          2|Priya|Mumbai| 28|
|          3|Rahul|  Pune| 30|
+-----------+-----+------+---+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)



## 34. Nested JSON

In [12]:
nested_json_file = base_path / "nested_customers.json"

nested_json_file.write_text(
'''{"customer_id": 1, "name": "Amit", "address": {"city": "Delhi", "state": "Delhi"}}
{"customer_id": 2, "name": "Priya", "address": {"city": "Mumbai", "state": "Maharashtra"}}
''',
encoding="utf-8"
)

nested_df = spark.read.json(str(nested_json_file))

nested_df.show(truncate=False)
nested_df.printSchema()

+---------------------+-----------+-----+
|address              |customer_id|name |
+---------------------+-----------+-----+
|{Delhi, Delhi}       |1          |Amit |
|{Mumbai, Maharashtra}|2          |Priya|
+---------------------+-----------+-----+

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)



## 35. Access and Flatten Nested JSON Fields

In [13]:
from pyspark.sql.functions import col, when, trim
nested_flat_df = nested_df.select(
    col("customer_id"),
    col("name"),
    col("address.city").alias("city"),
    col("address.state").alias("state")
)

nested_flat_df.show()

+-----------+-----+------+-----------+
|customer_id| name|  city|      state|
+-----------+-----+------+-----------+
|          1| Amit| Delhi|      Delhi|
|          2|Priya|Mumbai|Maharashtra|
+-----------+-----+------+-----------+



## 36. JSON with Array Field

In [13]:
orders_json_file = base_path / "orders_with_items.json"

orders_json_file.write_text(
'''{"order_id": 101, "customer": "Amit", "items": [{"product": "Laptop", "qty": 1}, {"product": "Mouse", "qty": 2}]}
{"order_id": 102, "customer": "Priya", "items": [{"product": "Mobile", "qty": 1}]}
''',
encoding="utf-8"
)

orders_nested_df = spark.read.json(str(orders_json_file))

orders_nested_df.show(truncate=False)
orders_nested_df.printSchema() # to get the names of nested columns, we have to check here

+--------+-------------------------+--------+
|customer|items                    |order_id|
+--------+-------------------------+--------+
|Amit    |[{Laptop, 1}, {Mouse, 2}]|101     |
|Priya   |[{Mobile, 1}]            |102     |
+--------+-------------------------+--------+

root
 |-- customer: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |-- order_id: long (nullable = true)



## 37. Explode JSON Array

In [14]:
from pyspark.sql.functions import explode

orders_items_df = orders_nested_df.select(
    "order_id",
    "customer",
    explode("items").alias("item")
).select(
    "order_id",
    "customer",
    col("item.product").alias("product"),
    col("item.qty").alias("quantity")
)

orders_items_df.show()

+--------+--------+-------+--------+
|order_id|customer|product|quantity|
+--------+--------+-------+--------+
|     101|    Amit| Laptop|       1|
|     101|    Amit|  Mouse|       2|
|     102|   Priya| Mobile|       1|
+--------+--------+-------+--------+



## 38. Corrupt JSON Records

In [15]:
bad_json_file = base_path / "bad_customers.json"

bad_json_file.write_text(
'''{"customer_id": 1, "name": "Amit"}
{"customer_id": 2, "name": "Priya"
{"customer_id": 3, "name": "Rahul"}
''',
encoding="utf-8"
)

bad_json_df = spark.read \
    .option("mode", "PERMISSIVE") \
    .json(str(bad_json_file))

bad_json_df.show(truncate=False)
bad_json_df.printSchema()

+----------------------------------+-----------+-----+
|_corrupt_record                   |customer_id|name |
+----------------------------------+-----------+-----+
|NULL                              |1          |Amit |
|{"customer_id": 2, "name": "Priya"|NULL       |NULL |
|NULL                              |3          |Rahul|
+----------------------------------+-----------+-----+

root
 |-- _corrupt_record: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)



Spark may create a special column such as `_corrupt_record` for malformed JSON.

## 39. Drop Malformed JSON Records

In [16]:
bad_json_drop_df = spark.read \
    .option("mode", "DROPMALFORMED") \
    .json(str(bad_json_file))

bad_json_drop_df.show()

+-----------+-----+
|customer_id| name|
+-----------+-----+
|          1| Amit|
|          3|Rahul|
+-----------+-----+



## 40. Reading Multiple JSON Files

In [17]:
json_folder = base_path / "json_daily"
json_folder.mkdir(exist_ok=True)

(json_folder / "customers_1.json").write_text(
'''{"customer_id": 1, "name": "Amit"}
{"customer_id": 2, "name": "Priya"}
''',
encoding="utf-8"
)

(json_folder / "customers_2.json").write_text(
'''{"customer_id": 3, "name": "Rahul"}
{"customer_id": 4, "name": "Sneha"}
''',
encoding="utf-8"
)

multi_json_df = spark.read \
                 .json(str(json_folder / "*.json"))

multi_json_df.show()

+-----------+-----+
|customer_id| name|
+-----------+-----+
|          3|Rahul|
|          4|Sneha|
|          1| Amit|
|          2|Priya|
+-----------+-----+



## 42. Write DataFrame as JSON

In [20]:
json_output_path = base_path / "output_customers_json"

customers_json_df.write \
    .mode("overwrite") \
    .json(str(json_output_path))

for file in json_output_path.iterdir():
    print(file.name)

.part-00000-32fa4ecf-2f6c-4a72-939b-842b5243b3f9-c000.json.crc
._SUCCESS.crc
part-00000-32fa4ecf-2f6c-4a72-939b-842b5243b3f9-c000.json
_SUCCESS


## 43. Write JSON with Compression

In [18]:
compressed_json_path = base_path / "compressed_json_gzip"

customers_json_df.write \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .json(str(compressed_json_path))

for file in compressed_json_path.iterdir():
    print(file.name)

.part-00000-5f86b013-5956-411a-830f-b9c9f2683600-c000.json.gz.crc
._SUCCESS.crc
part-00000-5f86b013-5956-411a-830f-b9c9f2683600-c000.json.gz
_SUCCESS


## 44. Write JSON Partitioned by Column

In [17]:
customers_json_df.show()

+---+------+-----------+-----+
|age|  city|customer_id| name|
+---+------+-----------+-----+
| 25| Delhi|          1| Amit|
| 28|Mumbai|          2|Priya|
| 30|  Pune|          3|Rahul|
+---+------+-----------+-----+



In [23]:
json_partitioned_path = base_path / "customers_json_by_city"

customers_json_df.write \
    .mode("overwrite") \
    .partitionBy("city") \
    .json(str(json_partitioned_path))

for item in sorted(json_partitioned_path.iterdir()):
    print(item.name)

._SUCCESS.crc
_SUCCESS
city=Delhi
city=Mumbai
city=Pune


In [24]:
single_json_output = base_path / "partition_customers_json_by_city"

customers_json_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .json(str(single_json_output))

for file in single_json_output.iterdir():
    print(file.name)



.part-00000-bc4460a1-5ce6-4891-be81-f50f25fd5b1c-c000.json.crc
._SUCCESS.crc
part-00000-bc4460a1-5ce6-4891-be81-f50f25fd5b1c-c000.json
_SUCCESS


In [22]:
single_json_output = base_path / "partition_customers_json_by_city"

customers_json_df.repartition(2) \
    .write \
    .mode("overwrite") \
    .json(str(single_json_output))

for file in single_json_output.iterdir():
    print(file.name)

.part-00000-3732ae89-2f93-4d47-9aa6-3432278159a6-c000.json.crc
.part-00001-3732ae89-2f93-4d47-9aa6-3432278159a6-c000.json.crc
._SUCCESS.crc
part-00000-3732ae89-2f93-4d47-9aa6-3432278159a6-c000.json
part-00001-3732ae89-2f93-4d47-9aa6-3432278159a6-c000.json
_SUCCESS


In [ ]:
spark.stop()